# shortcut-lens GPU runner

Thin launcher only -- **do not edit code in this notebook** (CLAUDE.md: notebooks are thin
launchers, all logic lives in `src/`). If a cell fails, fix the issue in the repo on your
laptop, push, and re-run this notebook from the new commit. See `docs/RUNBOOK_GPU.md` for the
full human checklist (one-time setup, secrets, quotas).

Set the two variables in the next cell (Claude Code's `/handoff` gives you both), then run every
cell in order.

In [ ]:
COMMIT = "<commit-sha-from-/handoff>"
JOBS = "jobs/<file>.yaml"
REPO_URL = "https://github.com/<owner>/shortcut-lens.git"
HF_ARTIFACT_REPO = "<user>/shortcut-lens-artifacts"

## 1. Clone the repo and check out the exact commit

In [ ]:
!git clone $REPO_URL repo
%cd repo
!git checkout $COMMIT

## 2. Install uv and sync the locked environment (CUDA extra)

In [ ]:
!curl -LsSf https://astral.sh/uv/install.sh | sh
!uv sync --extra cu126

## 3. Confirm the GPU is visible

In [ ]:
!nvidia-smi
!uv run python -c "import torch; print(torch.cuda.is_available(), torch.cuda.get_device_name(0))"

## 4. Log in to Hugging Face

`HF_TOKEN` must already be set as a notebook secret with access enabled (see
`docs/RUNBOOK_GPU.md`'s one-time setup) -- never paste the token into a cell.

In [ ]:
import os

# Kaggle: from kaggle_secrets import UserSecretsClient
# os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
# Colab: from google.colab import userdata
# os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
assert os.environ.get("HF_TOKEN"), "HF_TOKEN not set -- see docs/RUNBOOK_GPU.md setup"
!uv run huggingface-cli login --token $HF_TOKEN

## 5. Run the jobs file

Resumes automatically if this cell is re-run after a disconnect: each stage's own
config-hash/manifest check (or, for `train`, its epoch checkpoint) decides what is already done
(see `jobs.py`, ARCHITECTURE §6).

In [ ]:
!uv run slens run-jobs $JOBS

## 6. Push finished run folders to the Hugging Face Hub

Pushes every run directory that exists under `artifacts/runs/` -- back on your laptop,
`slens pull-artifacts --run-ids ... --repo-id ...` downloads them, then `slens validate-run
<run_id>` checks each one before use.

In [ ]:
import pathlib
import sys

sys.path.insert(0, "src")
from shortcut_lens.artifacts import HFHubStore

store = HFHubStore(repo_id=HF_ARTIFACT_REPO)
run_ids = [p.parent.name for p in pathlib.Path("artifacts/runs").glob("*/manifest.json")]
for run_id in run_ids:
    print(f"pushing {run_id} ...")
    store.push_run(run_id)
print(f"pushed {len(run_ids)} run(s): {run_ids}")